# Alexandria Trivia Game — Personality-Shaped

**What this notebook does:**
1. Loads a user's saved OCEAN personality profile from Firebase (written by
   the personality/recommendation notebook)
2. Fetches real facts about Alexandria & Egypt from a curated list of
   websites (tourism guides, Egypt's Ministry of Antiquities, museums,
   English + Arabic Wikipedia)
3. Asks an LLM to generate a fresh trivia game (mix of multiple-choice +
   short open-ended questions), where the *tone, framing, and difficulty*
   of the questions are shaped by the user's OCEAN profile — using the
   fetched facts as grounding so questions are factually real, not invented
4. Grades open-ended answers with the LLM, scores MCQs directly
5. Saves the game session/score back to Firebase
6. Exposes all of this as a FastAPI server tunneled via ngrok

**Endpoints:**

| Method | Endpoint              | Purpose                                         |
|--------|------------------------|--------------------------------------------------|
| POST   | `/game/new`            | Generate a new personality-shaped quiz for a user_id |
| POST   | `/game/grade`          | Submit answers → get per-question results + score |
| GET    | `/game/history/{user_id}` | Past game sessions for a user                |

**Setup required:**
- Same Groq API key as Notebook 1
- The **same** Firebase project/service account JSON as Notebook 1 (this notebook
  reads the profiles that Notebook 1 wrote)
- An ngrok authtoken (a *different* free tunnel than Notebook 1, since they're
  two separate Colab runtimes — you'll get a second public URL)

This notebook does NOT run the personality quiz itself — it assumes the user
already has a saved profile from Notebook 1. If no profile exists yet, the
`/game/new` endpoint returns a clear error asking the user to take the quiz first.

In [1]:
!pip install -q groq fastapi uvicorn pyngrok nest-asyncio firebase-admin beautifulsoup4 lxml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 5.9 MB/s eta 0:00:00


## 1. Configure secrets

Same approach as Notebook 1 — use Colab Secrets (🔑 icon in the sidebar):

- `GROQ_API_KEY`
- `NGROK_AUTH_TOKEN`
- `FIREBASE_CREDENTIALS_JSON` → **must be the same Firebase project as Notebook 1**,
  so this notebook can read the OCEAN profiles that were saved there.

In [2]:
import os
import json

def get_secret(name, prompt_text=None):
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            return val
    except Exception:
        pass
    from getpass import getpass
    return getpass(prompt_text or f"Enter {name}: ")

GROQ_API_KEY = get_secret("GROQ_API_KEY", "Enter your Groq API key: ")
NGROK_AUTH_TOKEN = get_secret("NGROK_AUTH_TOKEN", "Enter your ngrok authtoken: ")
FIREBASE_CREDENTIALS_JSON = get_secret(
    "FIREBASE_CREDENTIALS_JSON",
    "Paste your Firebase service account JSON (same project as Notebook 1): "
)

print("✅ Secrets loaded (values are hidden).")

✅ Secrets loaded (values are hidden).


In [3]:
from groq import Groq
import requests

client = Groq(api_key=GROQ_API_KEY)

# Models good at following instructions + producing valid JSON, best first.
# Small/specialised models (e.g. allam-2-7b, an Arabic 7B model) are NOT used:
# they produce broken JSON and invent facts, which was why /game/new failed.
PREFERRED = [
    "openai/gpt-oss-120b",
    "llama-3.3-70b-versatile",
    "moonshotai/kimi-k2-instruct-0905",
    "moonshotai/kimi-k2-instruct",
    "openai/gpt-oss-20b",
    "qwen/qwen3-32b",
    "meta-llama/llama-4-maverick-17b-128e-instruct",
    "meta-llama/llama-4-scout-17b-16e-instruct",
    "llama-3.1-8b-instant",
]

resp = requests.get(
    "https://api.groq.com/openai/v1/models",
    headers={"Authorization": f"Bearer {GROQ_API_KEY}"},
    timeout=15,
)
resp.raise_for_status()
available = {m["id"] for m in resp.json().get("data", [])}

# Ordered list of usable models (also used as the fallback chain on rate limits)
chat_models = [m for m in PREFERRED if m in available]
if not chat_models:
    raise RuntimeError(
        "None of the preferred Groq models are available on this key.\n"
        f"Available models: {sorted(available)}\n"
        "Add one of the strong chat models above to PREFERRED."
    )

selected_model = chat_models[0]
print(f"✅ Groq client ready. Using model: {selected_model}")
print(f"   Fallback chain: {chat_models[1:]}")

✅ Groq client ready. Using model: openai/gpt-oss-120b
   Fallback chain: ['openai/gpt-oss-20b']


In [4]:
import firebase_admin
from firebase_admin import credentials, firestore

cred_dict = json.loads(FIREBASE_CREDENTIALS_JSON)
cred = credentials.Certificate(cred_dict)

if not firebase_admin._apps:
    firebase_admin.initialize_app(cred)

db = firestore.client()
print("✅ Firebase Firestore connected.")
print(f"   Project: {cred_dict.get('project_id')}")

✅ Firebase Firestore connected.
   Project: rega-63259


In [5]:
from datetime import datetime, timezone
import uuid

def now_iso():
    return datetime.now(timezone.utc).isoformat()


def load_ocean_profile(user_id: str):
    """Reads the OCEAN profile saved by the Recommendation notebook. Returns None if not found."""
    snapshot = db.collection("users").document(user_id).get()
    return snapshot.to_dict() if snapshot.exists else None


def save_game_session(user_id: str, questions: list, score: int = None, total: int = None):
    """Saves a generated (and optionally graded) game session. Returns the session_id."""
    session_id = str(uuid.uuid4())
    db.collection("game_sessions").document(session_id).set({
        "user_id": user_id,
        "questions": questions,
        "score": score,
        "total": total,
        "created_at": now_iso(),
    })
    return session_id


def update_game_session_score(session_id: str, score: int, total: int, graded_questions: list):
    db.collection("game_sessions").document(session_id).update({
        "score": score,
        "total": total,
        "questions": graded_questions,
        "graded_at": now_iso(),
    })


def get_game_history(user_id: str, limit: int = 20):
    # where() + order_by() on different fields needs a Firestore composite index,
    # so filter in Firestore and sort in Python instead (works with zero setup).
    docs = db.collection("game_sessions").where("user_id", "==", user_id).stream()
    sessions = [doc.to_dict() | {"session_id": doc.id} for doc in docs]
    sessions.sort(key=lambda s: s.get("created_at", ""), reverse=True)
    return sessions[:limit]


print("✅ DB helpers ready: load_ocean_profile(), save_game_session(), "
      "update_game_session_score(), get_game_history()")

✅ DB helpers ready: load_ocean_profile(), save_game_session(), update_game_session_score(), get_game_history()


## 1.5. LLM helper

One function for every Groq call in this notebook:
- asks for **JSON mode** (`response_format={"type": "json_object"}`) so the model must return valid JSON
- on a rate limit or model error, **falls back** to the next model in `chat_models`
- strips Markdown code fences and parses the JSON safely

In [6]:
import re

def _parse_json(raw: str):
    """Parses LLM output into a dict; tolerates code fences and extra text around the JSON."""
    if raw is None:
        return None
    raw = raw.strip()
    if "```" in raw:
        m = re.search(r"```(?:json)?\s*(.*?)```", raw, re.S)
        if m:
            raw = m.group(1).strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        start, end = raw.find("{"), raw.rfind("}")
        if start != -1 and end > start:
            try:
                return json.loads(raw[start:end + 1])
            except json.JSONDecodeError:
                return None
    return None


def llm_call(messages, max_tokens=1500, temperature=0.5, json_mode=False):
    """
    Calls Groq, trying each model in chat_models (selected model first).
    Returns (text, model_used). Raises the last error if every model fails.
    """
    order = [selected_model] + [m for m in chat_models if m != selected_model]
    last_err = None
    for model in order:
        kwargs = dict(model=model, messages=messages,
                      max_tokens=max_tokens, temperature=temperature)
        if json_mode:
            kwargs["response_format"] = {"type": "json_object"}
        if model.startswith("openai/gpt-oss"):
            kwargs["reasoning_effort"] = "low"   # keep reasoning short so tokens go to the answer
        try:
            response = client.chat.completions.create(**kwargs)
            text = response.choices[0].message.content
            if text and text.strip():
                return text, model
            last_err = RuntimeError(f"{model} returned an empty response")
        except Exception as e:
            last_err = e
            print(f"  ⚠️  {model} failed ({type(e).__name__}): {str(e)[:150]} — trying next model")
    raise last_err


print("✅ LLM helper ready: llm_call(), _parse_json()")

✅ LLM helper ready: llm_call(), _parse_json()


## 2. Real-world content grounding (curated websites)

To make sure trivia questions are based on real facts (not LLM invention),
we pull short factual snippets from a curated list of websites about
Alexandria and Egypt (tourism sites, the Ministry of Antiquities, museum
pages, English and Arabic Wikipedia). A random subset of URLs is fetched
fresh each time a game is generated, so repeated games surface different
facts. Arabic-language sources are translated to English before use, so
trivia questions stay in one consistent language.

In [7]:
import requests
from bs4 import BeautifulSoup
import random


WEBSITE_URLS_PLACES = [
    "https://www.egypttoursplus.com/visiting-alexandria-egypt/",
    "https://www.bibalex.org/en/default",
    "https://egymonuments.gov.eg/en/archaeological-sites/alexandria",
    "https://egymonuments.gov.eg/en/monuments/the-cemetery-of-shatby",
    "https://egymonuments.gov.eg/en/monuments/qaitbay-fort",
    "https://egymonuments.gov.eg/en/museums/alexandria-national-museum",
    "https://egymonuments.gov.eg/en/collections/alexandria",
    "https://egymonuments.gov.eg/en/museums/graeco-roman-museum",
    "https://egymonuments.gov.eg/en/news/eliyahu-hanavi-synagogue-in-alexandria-reopened",
    "https://egymonuments.gov.eg/en/news/opening-of-the-qaitbay-citadel-visitor-center-in-alexandria",
    "https://egymonuments.gov.eg/en/sunken-monuments/abu-qir-bay",
    "https://egymonuments.gov.eg/en/news/visit-of-the-thai-foreign-minister-to-greco-roman-museum-in-alexandria",
    "https://egymonuments.gov.eg/en/archaeological-sites/kom-el-dikka",
    "https://egymonuments.gov.eg/en/sunken-monuments/shatby",
    "https://egymonuments.gov.eg/en/sunken-monuments/mamoura-bay",
    "https://egymonuments.gov.eg/en/archaeological-sites/abu-mina-1",
    "https://egymonuments.gov.eg/en/discover-egyptian-treasure",
    "https://en.wikipedia.org/wiki/Montaza_Palace",
    "https://en.wikipedia.org/wiki/Citadel_of_Qaitbay",
    "https://en.wikipedia.org/wiki/Ras_El_Tin_Palace",
    "https://www.egypttoursgate.com/blog/egypt-travel-news/alexandria-aquarium-museum",
    "https://en.wikipedia.org/wiki/Mahatet_El_Raml",
    "https://egymonuments.gov.eg/en/museums/royal-jewelry-museum",
    "https://en.wikipedia.org/wiki/Catacombs_of_Kom_El_Shoqafa",
    "https://ar.wikipedia.org/wiki/%D8%B9%D9%85%D9%88%D8%AF_%D8%A7%D9%84%D8%B3%D9%88%D8%A7%D8%B1%D9%8A",
    "https://en.wikipedia.org/wiki/Alexandria_National_Museum",
    "https://www.cairoopera.org/en/theaters/alexandria-opera-house-sayed-darwish-theatre/",
    "https://en.wikipedia.org/wiki/San_Stefano_Grand_Plaza",
    "https://ar.wikipedia.org/wiki/%D8%AC%D8%B1%D9%8A%D9%86_%D8%A8%D9%84%D8%A7%D8%B2%D8%A7",
    "https://en.wikipedia.org/wiki/Alexandria_Corniche",
    "https://ar.wikipedia.org/wiki/%D9%85%D8%B3%D8%AC%D8%AF_%D8%A3%D8%A8%D9%88_%D8%A7%D9%84%D8%B9%D8%A8%D8%A7%D8%B3_%D8%A7%D9%84%D9%85%D8%B1%D8%B3%D9%8A",
    "https://ar.wikipedia.org/wiki/%D9%86%D8%B5%D8%A8_%D8%A7%D9%84%D8%AC%D9%86%D8%AF%D9%8A_%D8%A7%D9%84%D9%85%D8%AC%D9%87%D9%88%D9%84_(%D8%A7%D9%84%D8%A5%D8%B3%D9%83%D9%86%D8%AF%D8%B1%D9%8A%D8%A9)",
    "https://www.bibalex.org/alexmed/gastronomy/Eateries/Details.aspx?id=OWVuuxxm5GiAtDAfe39TOw%3D%3D",
    "https://www.roughguides.com/egypt/alexandria-mediterranean-coast-delta/alexandria/",
    "https://www.roughguides.com/egypt/alexandria-mediterranean-coast-delta/",
]


def _looks_arabic(text: str) -> bool:
    """Rough check: does this text contain a meaningful amount of Arabic script?"""
    arabic_chars = sum(1 for ch in text if '\u0600' <= ch <= '\u06FF')
    return arabic_chars > 30  # a handful of Arabic words' worth


def _extract_main_text(html: str, max_chars: int = 1200) -> str:
    """
    Pulls readable factual text out of a real-world HTML page, dropping nav/
    header/footer/form/script noise. Prefers the meta description (usually a
    clean, pre-written summary) when present, falling back to the longest
    paragraph-like text blocks in the body otherwise.
    """
    soup = BeautifulSoup(html, "html.parser")

    # Strip obvious non-content elements before extracting anything
    for tag in soup(["script", "style", "nav", "header", "footer", "form", "noscript"]):
        tag.decompose()

    # 1. Prefer a meta description / og:description if it's substantial —
    #    these are usually clean, hand-written summaries on real sites.
    meta = soup.find("meta", attrs={"name": "description"}) or soup.find(
        "meta", attrs={"property": "og:description"}
    )
    if meta and meta.get("content") and len(meta["content"].strip()) > 60:
        text = meta["content"].strip()
    else:
        # 2. Fall back to paragraph-like text blocks, filtering short
        #    fragments (nav links, labels, buttons) that survived stripping.
        # Only <p> and <li> — NOT <div>, since a div's get_text() also
        # includes all its nested <p> text, which would double-count it.
        candidates = soup.find_all(["p", "li"])
        chunks = [c.get_text(" ", strip=True) for c in candidates]
        chunks = [c for c in chunks if len(c) > 80]  # drop short nav/label junk
        text = " ".join(chunks)

    text = " ".join(text.split())  # collapse whitespace
    return text[:max_chars]


def _fetch_one_pass(urls: list, timeout: int = 8) -> list:
    """Fetches the given URLs once each, returns whatever snippets succeed."""
    snippets = []
    for url in urls:
        try:
            resp = requests.get(
                url,
                timeout=timeout,
                headers={"User-Agent": "Mozilla/5.0 (AlexandriaTriviaBot/1.0)"},
            )
            resp.raise_for_status()

            soup_title = BeautifulSoup(resp.text, "html.parser").find("title")
            title = soup_title.get_text(strip=True) if soup_title else url

            text = _extract_main_text(resp.text)

            if len(text) < 80:
                print(f"  ⚠️  Skipped (too little usable text): {url}")
                continue

            if _looks_arabic(text):
                text = _translate_to_english(text)
                if text is None:
                    print(f"  ⚠️  Skipped (Arabic translation failed): {url}")
                    continue
                # Same page, so translate the title too — no need to re-check,
                # a page with an Arabic body almost certainly has an Arabic title.
                translated_title = _translate_to_english(title)
                title = translated_title or title

            snippets.append({"topic": title, "summary": text, "url": url})
            print(f"  ✅ Fetched: {title}")

        except Exception as e:
            print(f"  ⚠️  Skipped ({type(e).__name__}: {e}): {url}")
            continue

    return snippets


def fetch_website_snippets(n_sources: int = 6, timeout: int = 8, min_required: int = 2):
    """
    Picks n_sources random URLs from WEBSITE_URLS_PLACES, fetches each,
    extracts a clean factual snippet, and returns a list of
    {"topic": <page title>, "summary": <text>, "url": <url>} dicts.

    Logs a clear per-URL status line so failures are easy to diagnose,
    instead of silently degrading to an empty list. If fewer than
    min_required snippets succeed, retries once with a fresh sample drawn
    from the URLs not already tried (covers transient failures on a couple
    of unlucky picks without re-hitting the same dead URLs).
    """
    pool = list(WEBSITE_URLS_PLACES)
    chosen = random.sample(pool, min(n_sources, len(pool)))
    snippets = _fetch_one_pass(chosen, timeout=timeout)

    if len(snippets) < min_required:
        remaining = [u for u in pool if u not in chosen]
        if remaining:
            print(f"\n  Only {len(snippets)} snippet(s) succeeded — retrying with a fresh sample...\n")
            retry_n = min(n_sources, len(remaining))
            retry_urls = random.sample(remaining, retry_n)
            snippets += _fetch_one_pass(retry_urls, timeout=timeout)

    return snippets


def _translate_to_english(arabic_text: str) -> str:
    """
    Translates an Arabic factual snippet to English using Groq, so trivia
    questions stay in a single language. Returns None if translation fails
    (caller should skip that source rather than feed broken text to the game).
    """
    try:
        text, _ = llm_call(
            [
                {"role": "system",
                 "content": "Translate the given Arabic text to clear, natural English. "
                            "Return ONLY the translation, no commentary, no notes."},
                {"role": "user", "content": arabic_text},
            ],
            max_tokens=600,
            temperature=0.2,
        )
        return text.strip()
    except Exception:
        return None


print("✅ Website fact-fetcher ready: fetch_website_snippets()")

✅ Website fact-fetcher ready: fetch_website_snippets()


## 3. Personality-shaped question generation

The LLM is given:
- The user's OCEAN profile (scores + levels)
- Real snippets from the curated websites as factual grounding

It then writes a 6-question game: a mix of multiple-choice and short
open-ended questions. The **facts stay real** — only the tone, framing,
and difficulty are tailored to personality. For example:

- High **Openness** → questions about lesser-known history/art facts, framed
  with curiosity ("Here's something most visitors never learn about...")
- High **Extraversion** → framed as a fun social challenge / "impress your
  friends" trivia
- High **Neuroticism** → low-pressure framing, reassuring tone, no "you failed!"
  language
- Low **Conscientiousness** → quick, breezy questions, no long setup
- High **Agreeableness** → questions could be framed as something to share/discuss
  with others

In [8]:
GAME_SYSTEM_PROMPT = """
You are a fun trivia game designer for a tourism app about Alexandria and Egypt.

You will be given:
1. A visitor's OCEAN personality profile
2. A set of real factual snippets about Alexandria/Egypt (use ONLY these facts —
   do not invent historical facts, dates, or figures not present in the snippets)

Your job: write a 6-question trivia game using those facts. The questions must
be FACTUALLY GROUNDED in the snippets provided, but the TONE, FRAMING, and
DIFFICULTY should be shaped by the personality profile, per this guide:

- Openness HIGH: lean into surprising/lesser-known facts, curious tone
- Openness LOW: stick to well-known, mainstream facts
- Extraversion HIGH: frame questions as fun social challenges ("impress your friends")
- Extraversion LOW: calmer, more reflective framing
- Conscientiousness HIGH: more structured, precise questions (exact dates/names ok)
- Conscientiousness LOW: keep questions short, breezy, low-effort feeling
- Neuroticism HIGH: low-pressure, reassuring tone; NEVER say things like "you failed" or "wrong!" — frame incorrect answers gently
- Neuroticism LOW: tone can be more playful/competitive, light teasing is fine
- Agreeableness HIGH: frame some questions as fun facts to share with friends/family
- Agreeableness LOW: frame questions as personal challenges/achievements

RULES:
1. Produce exactly 6 questions: 4 multiple-choice, 2 short open-ended.
2. Multiple-choice questions need exactly 4 options with exactly one correct answer.
   The options are plain answer texts (no "A)" / "B." prefixes), and "correct_answer"
   must be copied EXACTLY, word for word, from one of the 4 options.
3. Open-ended questions should have a clear, short, gradeable correct answer (a name, date, place, or short phrase) — not essay-length.
4. Every question must be traceable to one of the provided factual snippets — do not use outside knowledge.
5. Return ONLY valid JSON, no text before or after, in this exact shape:

{
  "intro": "1-2 sentence personality-flavored intro to the game",
  "questions": [
    {
      "id": 1,
      "type": "mcq",
      "question": "...",
      "options": ["option text 1", "option text 2", "option text 3", "option text 4"],
      "correct_answer": "option text 2",
      "source_topic": "topic name from the snippets"
    },
    {
      "id": 2,
      "type": "open_ended",
      "question": "...",
      "correct_answer": "short expected answer",
      "source_topic": "topic name from the snippets"
    }
  ]
}
"""



_LETTER_PREFIX = re.compile(r"^\s*[A-Da-d][\).:\-]\s+")


def _validate_game(game) -> tuple:
    """
    Checks the LLM's game against the required shape and normalises it.
    Returns (clean_game, None) if valid, or (None, reason) if not.
    """
    if not isinstance(game, dict) or not isinstance(game.get("questions"), list):
        return None, "missing 'questions' list"

    clean_qs = []
    for i, q in enumerate(game["questions"], start=1):
        if not isinstance(q, dict):
            return None, f"question {i} is not an object"
        qtype = str(q.get("type", "")).lower().replace("-", "_").replace(" ", "_")
        question = str(q.get("question", "")).strip()
        answer = str(q.get("correct_answer", "")).strip()
        if not question or not answer:
            return None, f"question {i} is missing text or correct_answer"

        if qtype in ("mcq", "multiple_choice"):
            options = q.get("options") or q.get("answer_options")
            if not isinstance(options, list) or len(options) != 4:
                return None, f"question {i} must have exactly 4 options"
            options = [_LETTER_PREFIX.sub("", str(o)).strip() for o in options]
            # Accept a letter answer ("B") and convert it to the option text
            if len(answer) == 1 and answer.upper() in "ABCD":
                answer = options["ABCD".index(answer.upper())]
            answer = _LETTER_PREFIX.sub("", answer).strip()
            match = [o for o in options if o.lower() == answer.lower()]
            if not match:
                return None, f"question {i}: correct_answer is not one of the options"
            clean_qs.append({"id": i, "type": "mcq", "question": question,
                             "options": options, "correct_answer": match[0],
                             "source_topic": q.get("source_topic", "")})
        elif qtype in ("open_ended", "open", "short_answer"):
            clean_qs.append({"id": i, "type": "open_ended", "question": question,
                             "correct_answer": answer,
                             "source_topic": q.get("source_topic", "")})
        else:
            return None, f"question {i} has unknown type '{q.get('type')}'"

    n_mcq = sum(q["type"] == "mcq" for q in clean_qs)
    n_open = len(clean_qs) - n_mcq
    if len(clean_qs) != 6 or n_mcq != 4 or n_open != 2:
        return None, f"expected 4 mcq + 2 open_ended, got {n_mcq} mcq + {n_open} open_ended"

    return {"intro": str(game.get("intro", "")).strip(), "questions": clean_qs}, None


def generate_game(ocean_scores: dict, ocean_levels: dict, snippets: list, max_attempts: int = 3) -> dict:
    facts_block = "\n\n".join(
        f"TOPIC: {s['topic']}\n{s['summary']}" for s in snippets
    )

    profile_block = "\n".join(
        f"- {trait.capitalize()}: {float(ocean_scores[trait]):.1f}/5.0 [{ocean_levels[trait]}]"
        for trait in ["openness", "conscientiousness", "extraversion", "agreeableness", "neuroticism"]
    )

    user_message = f"""
Visitor OCEAN personality profile:
{profile_block}

Factual snippets to base questions on:
{facts_block}

Generate the 6-question trivia game now (4 "mcq" + 2 "open_ended"). Return ONLY the JSON.
"""
    messages = [
        {"role": "system", "content": GAME_SYSTEM_PROMPT},
        {"role": "user", "content": user_message},
    ]

    last_problem = None
    for attempt in range(1, max_attempts + 1):
        try:
            raw, model_used = llm_call(messages, max_tokens=3000, temperature=0.7, json_mode=True)
        except Exception as e:
            return {"error": f"All models failed: {e}"}

        game, problem = _validate_game(_parse_json(raw))
        if game:
            print(f"  ✅ Game generated with {model_used} (attempt {attempt})")
            return game

        last_problem = problem
        print(f"  ⚠️  Attempt {attempt}: invalid game ({problem}) — retrying...")
        # Tell the model exactly what was wrong so the next attempt fixes it
        messages = messages[:2] + [
            {"role": "assistant", "content": raw[:4000]},
            {"role": "user", "content": f"That JSON is invalid: {problem}. "
                                        "Return the corrected full JSON only, following the exact shape."},
        ]

    return {"error": f"Could not generate a valid game after {max_attempts} attempts: {last_problem}"}


print("✅ Game generator ready: generate_game()")

✅ Game generator ready: generate_game()


## 4. Grading

- **MCQ**: graded directly by string comparison (no LLM call needed — fast and free)
- **Open-ended**: graded by the LLM, since answers won't match the expected
  string exactly (typos, synonyms, partial phrasing). The LLM returns a
  simple correct/incorrect + short feedback per answer.

In [9]:
GRADING_SYSTEM_PROMPT = """
You are grading short open-ended trivia answers about Alexandria/Egypt.
For each question, you'll get the question, the expected correct answer,
and the user's actual answer. Decide if the user's answer is correct —
be lenient about spelling, phrasing, and partial answers that capture the
key fact (e.g. "Qaitbay" should match "Citadel of Qaitbay").

Return ONLY valid JSON in this shape, one entry per question, in the same order given:

{
  "results": [
    {"id": 1, "is_correct": true, "feedback": "short, encouraging, 1 sentence"},
    {"id": 2, "is_correct": false, "feedback": "short, encouraging, 1 sentence, gently states the correct answer"}
  ]
}
"""



def grade_open_ended(open_ended_items: list) -> dict:
    """
    open_ended_items: list of {"id", "question", "correct_answer", "user_answer"}
    Returns {"results": [{"id", "is_correct", "feedback"}, ...]}
    """
    if not open_ended_items:
        return {"results": []}

    items_block = "\n\n".join(
        f"Q{item['id']}: {item['question']}\n"
        f"Expected answer: {item['correct_answer']}\n"
        f"User's answer: {item['user_answer']}"
        for item in open_ended_items
    )

    try:
        raw, _ = llm_call(
            [
                {"role": "system", "content": GRADING_SYSTEM_PROMPT},
                {"role": "user", "content": items_block},
            ],
            max_tokens=1200,
            temperature=0.2,
            json_mode=True,
        )
    except Exception as e:
        return {"error": str(e), "results": []}

    parsed = _parse_json(raw) or {}
    results = []
    for r in parsed.get("results", []):
        try:
            rid = int(str(r.get("id")).lstrip("Qq"))   # model may return "Q5" or "5"
        except (TypeError, ValueError):
            continue
        is_correct = r.get("is_correct")
        if isinstance(is_correct, str):
            is_correct = is_correct.strip().lower() in ("true", "yes", "correct")
        results.append({"id": rid, "is_correct": bool(is_correct),
                        "feedback": r.get("feedback", "")})
    return {"results": results}


def grade_mcq(question: dict, user_answer: str) -> bool:
    """Accepts either the option letter (A-D) or the option text."""
    ans = (user_answer or "").strip()
    options = question.get("options", [])
    if len(ans) == 1 and ans.upper() in "ABCD" and len(options) == 4:
        ans = options["ABCD".index(ans.upper())]
    return ans.lower() == question["correct_answer"].strip().lower()


print("✅ Grading functions ready: grade_open_ended(), grade_mcq()")

✅ Grading functions ready: grade_open_ended(), grade_mcq()


## 5. FastAPI server

**POST `/game/new`**
```json
// request
{ "user_id": "abc123" }

// response
{
  "session_id": "...",
  "intro": "...",
  "questions": [
    {"id": 1, "type": "mcq", "question": "...", "options": [...]},
    {"id": 2, "type": "open_ended", "question": "..."}
  ]
}
```
Note: `correct_answer` is stripped out of the response sent to the frontend —
it's kept server-side (in Firestore) until grading time, so the answer key
can't leak through dev tools/network tab.

**POST `/game/grade`**
```json
// request
{
  "session_id": "...",
  "answers": [
    {"id": 1, "answer": "B"},
    {"id": 2, "answer": "Qaitbay"}
  ]
}

// response
{
  "score": 5,
  "total": 6,
  "results": [
    {"id": 1, "is_correct": true, "correct_answer": "B"},
    {"id": 2, "is_correct": true, "feedback": "Exactly right!"}
  ]
}
```

**GET `/game/history/{user_id}`** → list of past sessions with scores.

In [10]:
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import List, Dict, Any

app = FastAPI(title="Alexandria Personality-Shaped Trivia Game API")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],   # narrow this to your React app's domain before going to production
    allow_methods=["*"],
    allow_headers=["*"],
)


class NewGameRequest(BaseModel):
    user_id: str


class AnswerItem(BaseModel):
    id: int
    answer: str


class GradeRequest(BaseModel):
    session_id: str
    answers: List[AnswerItem]


def interpret_score_local(score: float) -> str:
    if score >= 3.5:
        return "High"
    elif score >= 2.5:
        return "Medium"
    return "Low"


def _strip_answers_for_client(questions: list) -> list:
    """Removes correct_answer/source_topic before sending questions to the frontend."""
    public = []
    for q in questions:
        clean = {"id": q["id"], "type": q["type"], "question": q["question"]}
        if q["type"] == "mcq":
            clean["options"] = q["options"]
        public.append(clean)
    return public


@app.post("/game/new")
def new_game(payload: NewGameRequest):
    profile = load_ocean_profile(payload.user_id)
    if profile is None:
        raise HTTPException(
            status_code=404,
            detail="No personality profile found for this user. Take the personality quiz first.",
        )

    ocean_scores = profile.get("ocean_scores")
    if not ocean_scores:
        raise HTTPException(status_code=404, detail="Profile has no OCEAN scores. Take the personality quiz first.")
    ocean_levels = profile.get("ocean_levels") or {
        t: interpret_score_local(v) for t, v in ocean_scores.items()
    }

    snippets = fetch_website_snippets(n_sources=6)
    if not snippets:
        raise HTTPException(status_code=503, detail="Could not fetch reference facts right now. Try again shortly.")

    game = generate_game(ocean_scores, ocean_levels, snippets)
    if "questions" not in game:
        raise HTTPException(status_code=502, detail=f"Game generation failed: {game.get('error', game)}")

    session_id = save_game_session(payload.user_id, game["questions"])

    return {
        "session_id": session_id,
        "intro": game.get("intro", ""),
        "questions": _strip_answers_for_client(game["questions"]),
    }


@app.post("/game/grade")
def grade_game(payload: GradeRequest):
    doc_ref = db.collection("game_sessions").document(payload.session_id)
    snapshot = doc_ref.get()
    if not snapshot.exists:
        raise HTTPException(status_code=404, detail="Session not found.")

    session = snapshot.to_dict()
    questions = {q["id"]: q for q in session["questions"]}
    user_answers = {a.id: a.answer for a in payload.answers}

    results = []
    open_ended_to_grade = []

    for qid, q in questions.items():
        user_answer = user_answers.get(qid, "")
        if q["type"] == "mcq":
            is_correct = grade_mcq(q, user_answer)
            results.append({
                "id": qid,
                "is_correct": is_correct,
                "correct_answer": q["correct_answer"],
            })
        else:
            open_ended_to_grade.append({
                "id": qid,
                "question": q["question"],
                "correct_answer": q["correct_answer"],
                "user_answer": user_answer,
            })

    if open_ended_to_grade:
        graded = grade_open_ended(open_ended_to_grade)
        graded_by_id = {r["id"]: r for r in graded.get("results", [])}
        for item in open_ended_to_grade:
            qid = item["id"]
            g = graded_by_id.get(qid, {"is_correct": False, "feedback": "Could not grade this answer."})
            results.append({
                "id": qid,
                "is_correct": bool(g.get("is_correct", False)),
                "feedback": g.get("feedback", ""),
                "correct_answer": questions[qid]["correct_answer"],
            })

    results.sort(key=lambda r: r["id"])
    score = sum(1 for r in results if r["is_correct"])
    total = len(results)

    update_game_session_score(payload.session_id, score, total, session["questions"])

    return {"score": score, "total": total, "results": results}


@app.get("/game/history/{user_id}")
def game_history(user_id: str):
    return {"sessions": get_game_history(user_id)}


@app.get("/health")
def health():
    return {"status": "ok", "model": selected_model}


print("✅ FastAPI app defined with routes: /game/new, /game/grade, /game/history/{user_id}, /health")

✅ FastAPI app defined with routes: /game/new, /game/grade, /game/history/{user_id}, /health


## 5.5. Test the game in-notebook (before exposing it with ngrok)

This cell plays the trivia game right here in Colab — no server, no ngrok,
no React needed yet. It uses `my_test_user`. If Notebook 1 hasn't saved a
real profile for that user, it seeds a fake OCEAN profile automatically so
you can test standalone (delete that seeding block once both notebooks are
wired together for real).

It walks through both flows:
1. `POST /game/new` — generates a 6-question game (4 MCQ + 2 open-ended)
2. You answer each question right in the cell's input prompts
3. `POST /game/grade` — MCQs graded by exact match, open-ended graded by the LLM
4. Prints your score and per-question feedback

Re-run any time to get a fresh set of questions.

In [12]:
# --- Interactive game: play it yourself right here in the notebook ---

from fastapi.testclient import TestClient
test_client = TestClient(app)

def run_interactive_game(user_id="my_test_user"):
    # Check there's a saved profile for this user (written by Notebook 1).
    # If you're testing this notebook standalone, this seeds a fake profile
    # so you don't need to run Notebook 1 first — delete the seeding block
    # once both notebooks are wired together for real.
    profile = load_ocean_profile(user_id)
    if profile is None:
        print(f"⚠️  No saved profile found for '{user_id}'. Seeding a fake one for this test run.\n")
        fake_scores = {"openness": 4.0, "conscientiousness": 3.0, "extraversion": 4.5,
                        "agreeableness": 3.5, "neuroticism": 2.0}
        db.collection("users").document(user_id).set({
            "ocean_scores": fake_scores,
            "ocean_levels": {t: interpret_score_local(v) for t, v in fake_scores.items()},
        })

    print("=" * 60)
    print("Generating your personality-shaped trivia game...")
    print("(this fetches real Alexandria/Egypt sources + calls Groq, may take a few seconds)")
    print("=" * 60 + "\n")

    r = test_client.post("/game/new", json={"user_id": user_id})
    if r.status_code != 200:
        print(f"❌ Error {r.status_code}: {r.text}")
        return None

    game = r.json()
    session_id = game["session_id"]

    print(game.get("intro", ""))
    print()

    user_responses = []
    for q in game["questions"]:
        print("─" * 60)
        if q["type"] == "mcq":
            print(f"Q{q['id']}. {q['question']}")
            for letter, opt in zip("ABCD", q["options"]):
                print(f"   {letter}) {opt}")
            print("   (type the letter or the answer)")
            while True:
                ans = input("Your answer: ").strip()
                if ans:
                    break
        else:  # open_ended
            print(f"Q{q['id']} (short answer). {q['question']}")
            while True:
                ans = input("Your answer: ").strip()
                if ans:
                    break
        user_responses.append({"id": q["id"], "answer": ans})
        print()

    print("─" * 60)
    print("Grading your answers...\n")

    r2 = test_client.post("/game/grade", json={"session_id": session_id, "answers": user_responses})
    if r2.status_code != 200:
        print(f"❌ Error {r2.status_code}: {r2.text}")
        return None

    graded = r2.json()

    print("=" * 60)
    print(f"   🎮  RESULTS: {graded['score']} / {graded['total']}")
    print("=" * 60 + "\n")

        # Map each question id -> what the user typed (turn "b" into the option text for MCQs)
    my_answers = {}
    for q, r in zip(game["questions"], user_responses):
        ans = r["answer"]
        if q["type"] == "mcq" and len(ans) == 1 and ans.upper() in "ABCD":
            ans = f"{ans.upper()}) {q['options']['ABCD'.index(ans.upper())]}"
        my_answers[q["id"]] = ans

    for res in graded["results"]:
        if res["is_correct"]:
            print(f"✅ Q{res['id']} — your answer: {my_answers[res['id']]}")
        else:
            print(f"❌ Q{res['id']} — your answer: {my_answers[res['id']]}")
            print(f"   ✔ correct answer: {res['correct_answer']}")
        if "feedback" in res:
            print(f"   {res['feedback']}")
        print()

    return graded


# Play it:
graded_result = run_interactive_game()

Generating your personality-shaped trivia game...
(this fetches real Alexandria/Egypt sources + calls Groq, may take a few seconds)

  ✅ Fetched: San Stefano Grand Plaza - Wikipedia
  ✅ Fetched: Abu Qir Bay - Discover Egypt's Monuments - Ministry of Tourism and Antiquities
  ⚠️  Skipped (too little usable text): https://www.bibalex.org/alexmed/gastronomy/Eateries/Details.aspx?id=OWVuuxxm5GiAtDAfe39TOw%3D%3D
  ✅ Fetched: Alexandria - Discover Egypt's Monuments - Ministry of Tourism and Antiquities
  ✅ Fetched: Abu al‑Abbas al‑Mursi Mosque – Wikipedia
  ✅ Fetched: Mahatet El Raml - Wikipedia
  ✅ Game generated with openai/gpt-oss-120b (attempt 1)
Ready to wow your friends with some cool Alexandria trivia? Let’s dive in and see how many fun facts you can share!

────────────────────────────────────────────────────────────
Q1. Which Greek architect designed the original layout of ancient Alexandria, linking the mainland to the island of Pharos?
   A) Dinocrates of Rhodes
   B) Vitruvius
  

## 6. Expose the API with ngrok

Run this **last**. It keeps running to serve requests — stop it with the ⏹ button when you're done.

In [ ]:
import nest_asyncio
from pyngrok import ngrok
import uvicorn
import time
import asyncio # Import asyncio for direct loop access

# Kill any existing ngrok processes before starting a new one
# This helps if ngrok was not properly shut down or if another instance is running
ngrok.kill()       # Forcefully kill ngrok processes
time.sleep(1)      # Give it a moment to ensure processes are terminated

nest_asyncio.apply()
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

public_url = ngrok.connect(8000).public_url
print("=" * 60)
print(f"🌍 PUBLIC GAME API URL: {public_url}")
print("=" * 60)
print("Share this URL with your React team. Example calls:")
print(f"  POST {public_url}/game/new")
print(f"  POST {public_url}/game/grade")
print(f"  GET  {public_url}/game/history/<user_id>")
print(f"  GET  {public_url}/health")
print("\nLeave this cell running to keep the API alive. Stop it with the ⏹ button.\n")

# Explicitly create and run the Uvicorn server to avoid asyncio.run() conflict
config = uvicorn.Config(app, host="0.0.0.0", port=8000, loop="asyncio")
server = uvicorn.Server(config)

loop = asyncio.get_event_loop()
loop.run_until_complete(server.serve())